# Paper 1 — Golden Age Semantic Reconfiguration

**Current stage: Phase 9 — Temporal Design Pre-registration and Feasibility**

Phase 8 established a validated primary chronology of **97 sonnets** across 6 authors and all 5 external historiographic stages. Phase 9 freezes that chronology and evaluates temporal designs **before any semantic feature or network is computed**.

This notebook compares fixed calendar windows and equal-count sensitivity windows under explicit date uncertainty, audits the external reference dates 1580 and 1605 without using them to place windows, and tests leave-one-author-out feasibility.

**No semantic representation, network edge, change-point statistic, or literary-historical label is used to choose the temporal design.**

In [ ]:
import re, shutil, subprocess, unicodedata, math
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES={
 'navarro_tei':('https://github.com/bncolorado/CorpusSonetosSigloDeOro.git','092a5fe70a4065a4d84bfed288bffd3851348f9c'),
 'gongora_scholarly':('https://github.com/gongoradigital/gongoraobra.git','3beadeecc059a7cc48499dc2683bb378a2630978'),
}
ROOT=Path('/content/gasr_phase9_sources'); ROOT.mkdir(exist_ok=True)

def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(['git','clone','--quiet',url,str(dst)],check=True)
    subprocess.run(['git','-C',str(dst),'checkout','--quiet',commit],check=True)
    got=subprocess.check_output(['git','-C',str(dst),'rev-parse','HEAD'],text=True).strip()
    assert got==commit,(name,got,commit)
    return dst

paths={k:clone(k,*v) for k,v in SOURCES.items()}
N=paths['navarro_tei']; G=paths['gongora_scholarly']; XML_ID='{http://www.w3.org/XML/1998/namespace}id'

def local(tag): return tag.split('}')[-1] if '}' in tag else tag
def el_text(el): return '' if el is None else ' '.join(' '.join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize('NFKD',str(s)); s=''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]','',s.lower())
def years_1580_1626(s):
    return sorted(set(int(x) for x in re.findall(r'(?<!\d)(1[56]\d{2})(?!\d)',str(s)) if 1580<=int(x)<=1626))

rows=[]
for fp in sorted(N.rglob('*.xml')):
    root=ET.parse(fp).getroot(); lines=[el_text(x) for x in root.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    txt='\n'.join(lines); author=fp.parent.name
    rows.append({'n_id':f'{author}::{fp.name}','author_dir':author,'text':txt,'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'first_line':lines[0]})
n=pd.DataFrame(rows); assert len(n)==5078,len(n)
print('Pinned sources ready')
print('Navarro poems:',len(n),'| author folders:',n.author_dir.nunique())

In [ ]:
# Freeze the verified Phase-8 PRIMARY chronology only.
# Non-primary circulation/sensitivity channels remain documented in earlier phases and are not needed to design composition-time windows.
primary_rows=[]
def add(pid,author,lo,hi,confidence,basis): primary_rows.append({'n_id':pid,'author_dir':author,'composition_min':int(lo),'composition_max':int(hi),'temporal_confidence':confidence,'temporal_basis':basis})

# Góngora: reproduce Phase 4/5 one-to-one linkage to the pinned scholarly chronology.
groot=ET.parse(G/'gongora_obra-poetica.xml').getroot(); parent={child:par for par in groot.iter() for child in par}; grows=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,'')
    if local(el.tag)!='div' or not xid.lower().startswith('poem'): continue
    lines=[el_text(x) for x in el.iter() if local(x.tag)=='l']; lines=[x for x in lines if x]
    if not lines: continue
    vals=[]; cur=el
    for _ in range(6):
        vals+=list(cur.attrib.values())
        if cur.text: vals.append(cur.text)
        for ch in list(cur):
            if local(ch.tag) in {'head','date','label'}: vals.append(el_text(ch))
            if ch.tail: vals.append(ch.tail)
        cur=parent.get(cur)
        if cur is None: break
    ys=sorted(set(y for v in vals for y in years_1580_1626(v))); txt='\n'.join(lines)
    grows.append({'g_id':xid,'n_lines':len(lines),'signature':norm(txt),'first2_signature':norm('\n'.join(lines[:2])),'scholarly_year':ys[0] if len(ys)==1 else pd.NA,'year_status':'unique' if len(ys)==1 else ('ambiguous' if len(ys)>1 else 'missing')})
g=pd.DataFrame(grows); g14=g[(g.n_lines==14)&g.signature.ne('')].copy(); g_by_id=g.set_index('g_id',drop=False); ng=n[n.author_dir.eq('Gongora')].copy(); sig_to_gids=g14.groupby('signature').g_id.apply(list).to_dict(); links=[]
for r in ng.itertuples(index=False):
    exact_ids=sig_to_gids.get(r.signature,[])
    if len(exact_ids)==1: gid,score,method=exact_ids[0],1.0,'exact'
    else:
        best_gid,best_score=None,-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best_score: best_gid,best_score=gr.g_id,sc
        gid,score,method=best_gid,best_score,'fuzzy'
    links.append({'n_id':r.n_id,'g_id':gid,'method':method,'score':float(score),'preaccept':method=='exact' or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept].copy(); collisions=set(pre.g_id.value_counts()[lambda s:s>1].index); glink['accept_phase4']=glink.preaccept&~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[['g_id','scholarly_year','year_status']],on='g_id',how='left'); acc=acc[acc.year_status.eq('unique')&acc.scholarly_year.notna()].copy()
for r in acc.itertuples(index=False): add(r.n_id,'Gongora',r.scholarly_year,r.scholarly_year,'A' if r.method=='exact' else 'B','scholarly_chronology_year_exact_link' if r.method=='exact' else 'scholarly_chronology_year_fuzzy_link')
phase4_nids=set(acc.n_id); phase4_gids=set(acc.g_id); unmatched=ng[~ng.n_id.isin(phase4_nids)].copy(); first2_index=g14.groupby('first2_signature').g_id.apply(list).to_dict(); recovered=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    if len(ids2)!=1: continue
    gid=ids2[0]
    if gid in phase4_gids: continue
    gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio()
    if score>=0.95 and gr.year_status=='unique' and pd.notna(gr.scholarly_year): recovered.append((r.n_id,gid,score,int(gr.scholarly_year)))
rec=pd.DataFrame(recovered,columns=['n_id','g_id','score','year']); dup=set(rec.g_id.value_counts()[lambda s:s>1].index) if len(rec) else set(); rec=rec[~rec.g_id.isin(dup)]
for r in rec.itertuples(index=False): add(r.n_id,'Gongora',r.year,r.year,'B','scholarly_chronology_year_variant_link')
assert sum(x['author_dir']=='Gongora' for x in primary_rows)==58

# Garcilaso: verified Phase-4 seed.
GAR={**{i:(1526,1532,'B','scholarly_phase_interval') for i in [1,2,3,4,6,26,27]},25:(1534,1535,'B','scholarly_interval'),33:(1535,1535,'A','historically_anchored_scholarly_year'),35:(1535,1535,'A','historically_anchored_scholarly_year'),**{i:(1533,1535,'B','revised_scholarly_interval') for i in [7,8,12,15,19,28,30,31]}}
for no,(lo,hi,conf,basis) in GAR.items(): add(f'GarcilasoDeLaVega::GarcilasoDeLaVega_{no}.xml','GarcilasoDeLaVega',lo,hi,conf,basis)

# Phase 6–8 verified poem-level anchors.
for no,lo,hi,conf,basis in [(30,1596,1596,'B','Cadiz_1596'),(13,1598,1598,'A','FelipeII_tomb_1598'),(31,1597,1598,'B','Herrera_death_epitaph')]: add(f'Cervantes::Cervantes_{no}.xml','Cervantes',lo,hi,conf,basis)
for no,lo,hi,basis in [(224,1574,1574,'Alameda_CarlosV'),(279,1578,1579,'Barahona_Granada'),(276,1573,1574,'Bazan_Tunis'),(300,1580,1582,'Portugal_to_H'),(281,1578,1578,'DonJuan_de_Austria')]: add(f'FernandoDeHerrera::FernandoDeHerrera_{no}.xml','FernandoDeHerrera',lo,hi,'B',basis)
for no in [2,19,4,5]: add(f'PedroEspinosa::PedroEspinosa_{no}.xml','PedroEspinosa',1594,1596,'B','Espinosa_happiness_period_1594_1596')
for no,year,basis in [(131,1609,'Carrillo_sonnet_1609'),(69,1611,'Aminta_1611'),(70,1611,'Aminta_1611'),(72,1611,'Aminta_1611'),(76,1611,'Aminta_1611'),(42,1610,'HenryIV_1610'),(43,1610,'HenryIV_1610'),(45,1610,'HenryIV_1610'),(44,1624,'Osuna_1624')]: add(f'Quevedo::Quevedo_{no}.xml','Quevedo',year,year,'B',basis)

primary=pd.DataFrame(primary_rows).drop_duplicates('n_id').copy(); known=set(n.n_id); missing=sorted(set(primary.n_id)-known); assert not missing,missing[:10]
counts=primary.groupby('author_dir').size().to_dict(); expected={'Gongora':58,'GarcilasoDeLaVega':18,'Quevedo':9,'FernandoDeHerrera':5,'PedroEspinosa':4,'Cervantes':3}
assert len(primary)==97 and counts==expected,(len(primary),counts)
print('PHASE-8 PRIMARY CHRONOLOGY FROZEN: 97 poems')
display(pd.Series(counts).sort_values(ascending=False).rename('primary_poems').to_frame())

## Phase 9 design freeze

For every bounded primary interval \( [a_i,b_i] \), the temporal-design audit samples an integer year from a **discrete uniform distribution** on the admitted interval. Exact-year poems remain fixed. No semantic result is available at this stage.

Fixed calendar candidates use widths **10, 15, 20, 25 years** and a **5-year step**. Equal-count windows of **20, 25, 30 poems** are evaluated only as sensitivity designs.

The dates **1580** and **1605** are external historiographic references only: windows are constructed first and the markers are audited afterward.

A fixed window is labelled operationally usable in a Monte Carlo realization when it has at least 10 poems, at least 2 authors, effective authors \(N_{eff}\ge1.5\), and largest-author share \(\le0.85\). These are feasibility guardrails, not inferential or literary thresholds.

In [ ]:
primary=primary.copy(); primary['composition_min']=primary.composition_min.astype(int); primary['composition_max']=primary.composition_max.astype(int); primary['interval_width']=primary.composition_max-primary.composition_min
interval_summary=pd.DataFrame([{'primary_poems':len(primary),'exact_year_poems':int((primary.interval_width==0).sum()),'bounded_interval_poems':int((primary.interval_width>0).sum()),'max_interval_width':int(primary.interval_width.max()),'median_interval_width':float(primary.interval_width.median()),'min_year':int(primary.composition_min.min()),'max_year':int(primary.composition_max.max())}]); display(interval_summary)

SEED=20260825; MC_DRAWS=1000; WIDTHS=[10,15,20,25]; STEP=5; rng=np.random.default_rng(SEED)
lo=primary.composition_min.to_numpy(int); hi=primary.composition_max.to_numpy(int); authors=primary.author_dir.to_numpy(); sampled_years=np.empty((MC_DRAWS,len(primary)),dtype=int)
for j,(a,b) in enumerate(zip(lo,hi)): sampled_years[:,j]=a if a==b else rng.integers(a,b+1,size=MC_DRAWS)

def metrics(mask):
    idx=np.flatnonzero(mask)
    if len(idx)==0: return 0,0,0.0,1.0
    vals,cts=np.unique(authors[idx],return_counts=True); p=cts/cts.sum(); neff=float(np.exp(-(p*np.log(p)).sum())); return int(len(idx)),int(len(vals)),neff,float(p.max())

domain_start=(int(lo.min())//5)*5; domain_end=int(math.ceil(int(hi.max())/5)*5); rows=[]
for width in WIDTHS:
    for start in range(domain_start,domain_end-width+2,STEP):
        end=start+width-1
        for m in range(MC_DRAWS):
            npo,na,neff,top=metrics((sampled_years[m]>=start)&(sampled_years[m]<=end)); usable=(npo>=10 and na>=2 and neff>=1.5 and top<=0.85); rows.append((width,start,end,m,npo,na,neff,top,usable))
fixed_mc=pd.DataFrame(rows,columns=['width','start','end','draw','n_poems','n_authors','effective_authors','top_author_share','usable'])
fixed_support=(fixed_mc.groupby(['width','start','end']).agg(usable_probability=('usable','mean'),n_poems_median=('n_poems','median'),n_poems_q10=('n_poems',lambda x:x.quantile(.10)),authors_median=('n_authors','median'),effective_authors_median=('effective_authors','median'),effective_authors_q10=('effective_authors',lambda x:x.quantile(.10)),top_share_median=('top_author_share','median'),top_share_q90=('top_author_share',lambda x:x.quantile(.90))).reset_index()); fixed_support['stable_usable']=fixed_support.usable_probability.ge(.80)

def longest_run(g):
    cur=best=0
    for v in g.sort_values('start').stable_usable: cur=cur+1 if v else 0; best=max(best,cur)
    return best
summary=[]
for width,g in fixed_support.groupby('width'):
    stable=g[g.stable_usable]; summary.append({'width':int(width),'step':STEP,'n_windows':len(g),'stable_usable_windows':int(g.stable_usable.sum()),'stable_fraction':float(g.stable_usable.mean()),'longest_consecutive_stable_windows':int(longest_run(g)),'first_stable_start':pd.NA if stable.empty else int(stable.start.min()),'last_stable_end':pd.NA if stable.empty else int(stable.end.max()),'median_window_effective_authors':float(g.effective_authors_median.median()),'median_window_top_share':float(g.top_share_median.median())})
fixed_design_summary=pd.DataFrame(summary).sort_values(['stable_usable_windows','width'],ascending=[False,True])
print('FIXED-WINDOW DESIGN SUMMARY'); display(fixed_design_summary)
print('STABLE SUPPORTED WINDOWS (P[usable] >= .80)'); display(fixed_support[fixed_support.stable_usable].sort_values(['width','start']).head(80))

In [ ]:
# External markers: audit after window construction.
marker_rows=[]
for marker in [1580,1605]:
    for width,g in fixed_support.groupby('width'):
        z=g[(g.start<=marker)&(g.end>=marker)]; marker_rows.append({'marker':marker,'width':int(width),'candidate_windows_containing_marker':len(z),'best_usable_probability':float(z.usable_probability.max()) if len(z) else np.nan,'median_usable_probability':float(z.usable_probability.median()) if len(z) else np.nan,'best_effective_authors_median':float(z.effective_authors_median.max()) if len(z) else np.nan,'lowest_top_share_median':float(z.top_share_median.min()) if len(z) else np.nan})
marker_support=pd.DataFrame(marker_rows); print('EXTERNAL-MARKER SUPPORT AUDIT'); display(marker_support)

# Equal-count windows: sensitivity only.
eq=[]
for k in [20,25,30]:
    vals=[]
    for m in range(MC_DRAWS):
        order=np.argsort(sampled_years[m],kind='stable'); ys=sampled_years[m,order]
        for s in range(0,len(order)-k+1,5):
            idx=order[s:s+k]; c=np.unique(authors[idx],return_counts=True)[1]; p=c/c.sum(); vals.append((int(ys[s+k-1]-ys[s]),len(c),float(np.exp(-(p*np.log(p)).sum())),float(p.max())))
    d=pd.DataFrame(vals,columns=['year_span','n_authors','effective_authors','top_author_share']); eq.append({'k':k,'stride':5,'windows_evaluated':len(d),'median_year_span':float(d.year_span.median()),'q90_year_span':float(d.year_span.quantile(.90)),'median_authors':float(d.n_authors.median()),'median_effective_authors':float(d.effective_authors.median()),'q90_top_author_share':float(d.top_author_share.quantile(.90)),'share_windows_2plus_authors':float((d.n_authors>=2).mean())})
equal_count_summary=pd.DataFrame(eq); print('EQUAL-COUNT SENSITIVITY SUMMARY'); display(equal_count_summary)

# Leave-one-author-out feasibility.
loo=[]
for excluded in sorted(primary.author_dir.unique()):
    keep=authors!=excluded
    for width in WIDTHS:
        probs=[]
        for start in range(domain_start,domain_end-width+2,STEP):
            end=start+width-1; u=[]
            for m in range(MC_DRAWS):
                npo,na,neff,top=metrics(keep&(sampled_years[m]>=start)&(sampled_years[m]<=end)); u.append(npo>=10 and na>=2 and neff>=1.5 and top<=0.85)
            probs.append(float(np.mean(u)))
        loo.append({'excluded_author':excluded,'width':width,'stable_usable_windows_p80':int(sum(p>=.80 for p in probs)),'mean_window_usable_probability':float(np.mean(probs)),'max_window_usable_probability':float(np.max(probs))})
loo_summary=pd.DataFrame(loo); print('LEAVE-ONE-AUTHOR-OUT FEASIBILITY'); display(loo_summary.sort_values(['excluded_author','width']))

In [ ]:
OUT=Path('/content/gasr_phase9_outputs'); OUT.mkdir(exist_ok=True)
primary.to_csv(OUT/'phase9_primary_chronology.csv',index=False); interval_summary.to_csv(OUT/'phase9_interval_summary.csv',index=False); fixed_design_summary.to_csv(OUT/'phase9_fixed_window_design_summary.csv',index=False); fixed_support.to_csv(OUT/'phase9_fixed_window_support.csv',index=False); marker_support.to_csv(OUT/'phase9_external_marker_support.csv',index=False); equal_count_summary.to_csv(OUT/'phase9_equal_count_sensitivity_summary.csv',index=False); loo_summary.to_csv(OUT/'phase9_leave_one_author_out_feasibility.csv',index=False)
assert len(primary)==97 and primary.author_dir.nunique()==6; assert set(fixed_design_summary.width)=={10,15,20,25}; assert set(marker_support.marker)=={1580,1605}; assert set(equal_count_summary.k)=={20,25,30}
print('\nPHASE 9 CHECKPOINT'); print('------------------'); print('Primary chronology frozen:',len(primary),'poems |',primary.author_dir.nunique(),'authors'); print('Monte Carlo draws:',MC_DRAWS,'| seed:',SEED); print('Fixed widths tested:',WIDTHS,'| step:',STEP); print('Equal-count sensitivity k:',equal_count_summary.k.tolist()); print('NO SEMANTIC FEATURES OR NETWORKS COMPUTED'); print('Top fixed designs by stable support:'); display(fixed_design_summary.head(4)); print('Outputs:',OUT)